# A2-Lite — one protected feature-KD pilot
Latest revision: **A2-LITE-PROTECTED-FEATURE-KD-2026-10-15-r1**.

Question: can per-step safeguards preserve Pedestrian performance while retaining useful teacher guidance? This is ONE bounded recipe, not a sweep. Same A2-Lite E1/D1,640x192,50 queries; original untrained pruned initialization; same512 TRAIN frames and five fixed epochs (2,560 updates per arm). Train a fresh label-only control and GT + protected-feature-KD arm. Never initialize either arm from a recovered or tiny-fit checkpoint.

Labels remain unchanged for both classes. Reuse the exact original512 teacher feature files: no teacher model loading, inference, new cache or unsafe pickle fallback. The scheduled historical coefficient is only an upper ceiling with the same128-step warmup. At EVERY KD step, measure gradients for GT total, exact Pedestrian/Vehicle GT shares and combined feature guidance on identical trainable coordinates. Class GT shares retain global matching, normalization, dimension compensation and class-channel negatives.

Skip the ENTIRE combined teacher term if it opposes either class or total GT in any checked group (all trainable, visual encoder, late backbone, other trainable; cosine roundoff tolerance1e-6). Otherwise select one detached nonnegative coefficient capped so teacher gradient norm is at most5% of GT total in every group. No GT gradient projection, class reweighting or redistribution. These safeguards apply BEFORE global clipping and AdamW; they do NOT guarantee improved AP or class preservation.

Only512 TRAIN images are staged before training, with four I/O workers and hash-verified local reuse. Stage3769 VAL images only after both training arms complete. Choose a GPU (A100 preferred), open this fresh notebook and run sections1–7. After a reset rerun all sections to restore private source/data; same-runtime checkpoints resume. If a prepared run's GPU/software differs, keep it intact and choose a new RUN_ID instead of mixing runs. No old CUDA stack or native extension build.

Fixed epoch5 is evaluated on all3769 VAL images. Success requires all five AP metrics nonregressing, mean moderate3D AP gain>=0.1 point, and nearby recall drop<=2 percentage points per class. Nearby recall uses2D IoU>=0.5, score>=0.001, Vehicle<=40m/Pedestrian<=30m; it is not3D correctness. No full-data run, coefficient sweep or deployment follows automatically.

## 1. Drive, paths and all helpers
Only edit paths if your Drive layout differs. Completed runs are read-only. This pilot needs the original student initialization and saved teacher features, not teacher or endpoint checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-PROTECTED-FEATURE-KD-2026-10-15-r1'
RUN_ID='a2_lite_512_protected_feature_kd_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
COMPLETED_RUN=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_features/a2_lite_512_foreground_feature_kd_r1')
FEATURE_RUN=COMPLETED_RUN
GRADIENT_AUDIT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/diagnostics/a2_lite_feature_gradients/a2_lite_feature_endpoint_gradients_r1')
FEATURE_FILES=COMPLETED_RUN/'teacher_features'
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
COMPACT=None  # Optional original640 export folder/ZIP; default uses the reviewed run's student_export.
SOURCE=Path('/content/MonoDETR_A2LITE_PROTECTED_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_protected_'+RUN_ID)
PREVIOUS_DATA=Path('/content/kitti_a2_lite_accuracy_feature_a2_lite_512_foreground_feature_kd_r1')
AUDIT_DATA=Path('/content/kitti_a2_lite_accuracy_feature_audit_a2_lite_feature_endpoint_gradients_r1')
CANDIDATES=[PREVIOUS_DATA,AUDIT_DATA,Path('/content/kitti_a2_lite_accuracy_cache_prepare_v1'),Path('/content/kitti'),Path('/content/drive/MyDrive/datasets/kitti')]
FEATURE_CANDIDATES=[PREVIOUS_DATA/'teacher_feature_cache',Path('/content/a2_lite_feature_audit_weights_a2_lite_feature_endpoint_gradients_r1/teacher_features')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_protected_features')/RUN_ID
for protected in (COMPLETED_RUN,GRADIENT_AUDIT,INITIALIZATION.parent):
    if OUTPUT.resolve().is_relative_to(protected.resolve()):raise RuntimeError('Use a separate protected-pilot output')
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/recover_a2_lite_protected_features.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;PREPARED=False;QUALIFIED=False;TRAINED=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True);tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nFresh GT control versus per-step protected feature KD.\nOutput:',OUTPUT)


## 2. Update main and use the current GPU
Install ordinary dependencies only; leave Colab's Torch/CUDA unchanged. A fresh matched pair may use the current runtime; its GPU/software becomes frozen for resume. No teacher checkpoint or old training virtual environment is required.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use clean main.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Open the latest protected-feature-KD notebook from main')
for path in (FEATURE_RUN,GRADIENT_AUDIT,INITIALIZATION):
    if not path.exists():raise FileNotFoundError(f'{path}: correct section1 paths')
if not any(p.is_dir() for p in [FEATURE_FILES,*FEATURE_CANDIDATES]):raise FileNotFoundError('Need the saved512 teacher_features from the completed run; results ZIPs exclude feature tensors')
packages=['numpy==2.1.3','timm==1.0.27','onnxruntime==1.30.0','pillow==11.3.0','scipy==1.16.3','opencv-python-headless==5.0.0.93','pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'ordinary_dependencies')
run_logged([PYTHON,'-c',"import torch;assert torch.cuda.is_available(),'Select a GPU runtime';torch.cuda.init();print(torch.__version__,torch.cuda.get_device_name(0),'portable training; no native extension')"],'gpu_probe')
READY=True


## 3. Tests, private source restore and efficient TRAIN-only staging
Review the exact failed-feature run and completed gradient audit, pin the original initialization, copy/hash only512 training frames with four workers, and reuse the original512 detached teacher features. Restore source before validating an existing run. No validation images are staged yet.


In [ ]:
PREPARED=False
if not READY:raise RuntimeError('Run sections1 and2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_protected_features.py','-v'],'protected_tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
for candidate in FEATURE_CANDIDATES:extra += ['--feature-candidate',candidate]
if COMPACT is not None:extra += ['--compact',COMPACT]
stage('prepare','--feature-run',FEATURE_RUN,'--gradient-audit',GRADIENT_AUDIT,'--features-dir',FEATURE_FILES,
      '--initialization',INITIALIZATION,'--repo',SOURCE,'--dataset',DATASET,*extra)
MANIFEST=json.loads((OUTPUT/'protected_manifest.json').read_text())
assert len(MANIFEST['pilot_ids'])==512 and MANIFEST['recipe']['epochs']==5
assert MANIFEST['recipe']['feature_gradient_cap']==0.05 and MANIFEST['architecture_changed'] is False
PREPARED=True
print('TRAIN512 staged; VAL3769 staging deferred. Teacher model forward passes:0.')
print('Scheduled coefficient ceiling:',MANIFEST['scheduled_max_coefficient'])
print('Runtime:',MANIFEST['runtime'])


## 4. Learnability and zero-update safeguard preflight
Student parity smoke uses a disposable model for two diagnostic updates; the16-frame/128-update label-only tiny fit is also diagnostic and NEVER initializes either main arm. The32-frame guard smoke performs ZERO optimizer updates, verifies unchanged weights/buffers and checks that some teacher guidance is accepted for both classes. Stop if any preflight fails.


In [ ]:
QUALIFIED=False
if not PREPARED:raise RuntimeError('Run sections1–3 first')
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text())
if TINY['passed']:
    stage('guard-smoke')
    CHECK=json.loads((OUTPUT/'protected_smoke.json').read_text())
    assert CHECK['optimizer_steps']==0 and CHECK['weights_buffers_unchanged']
    QUALIFIED=bool(CHECK['passed'])
    print('Accepted teacher objects in32-frame preflight:',CHECK['accepted_teacher_object_counts'])
    print('Guard preflight passed:',QUALIFIED)
else:print('STOP: label-only tiny learnability check failed. Return section7 evidence.')


## 5. Train one fresh matched pair
Exactly2,560 main updates per arm. Protection adds gradient-measurement work only to training; inference architecture is unchanged. Checkpoints every128 updates include policy, runtime, guard counters and optimizer state. At most127 interrupted updates repeat after resume. No model is selected on validation.


In [ ]:
TRAINED=False
if QUALIFIED:
    stage('train','--role','control')
    stage('train','--role','kd')
    for role in ('control','kd'):
        report=json.loads((OUTPUT/role/'training_complete.json').read_text())
        assert report['optimizer_steps']==2560 and report['tiny_checkpoint_used'] is False
        print(role,'main updates:',report['optimizer_steps'],'guard statistics:',report['guard_statistics'])
    TRAINED=True
else:print('No paired training started; return section7 preflight evidence.')


## 6. Stage VAL only now; evaluate fixed epoch5
Both arms must finish before staging3769 VAL frames. Evaluate the same fixed endpoint and use the unchanged AP/nearby-recall guardrails. Count accepted/skipped/capped updates, cap violations and accepted teacher objects for both classes. A technically enforced gradient rule does not guarantee an accuracy benefit. No automatic next variant, full training or deployment.


In [ ]:
if TRAINED:
    validation_candidates=[]
    for candidate in CANDIDATES:validation_candidates += ['--candidate',candidate]
    stage('stage-validation',*validation_candidates)
    stage('evaluate','--role','control')
    stage('evaluate','--role','kd')
    stage('compare')
    REPORT=json.loads((OUTPUT/'protected_comparison.json').read_text())
    for role,result in REPORT['results'].items():
        print(role,json.dumps({'metrics':result['metrics'],'nearby_2d_recall':{c:result['nearby']['classes'][c]['near_recall'] for c in ('Vehicle','Pedestrian')}},indent=2))
    print(json.dumps({k:REPORT[k] for k in ('versus_label_only','runtime_guards_passed','benefit_guardrail_passed','guard_statistics','next_step')},indent=2))
else:print('Validation skipped: no complete matched training pair.')


## 7. Download results, including after failure
Return **a2_lite_protected_feature_results.zip**. Reports include historical evidence, guard preflight, training counters and sampled per-step guard decisions, fixed-endpoint evaluations and combined logs. Weights, feature tensors, packages and images stay on disk and are excluded. If an earlier section fails, run this section to return the logs.


In [ ]:
BUNDLE=OUTPUT/'a2_lite_protected_feature_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Helper missing; package readable evidence directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable logs and reports.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(OUTPUT).parts and path.suffix in ('.json','.csv','.txt','.log'):
                archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
